# Lab 3 — Building a RAG System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/angeloziletti/nlp-cas-dsmh/blob/main/notebooks/lab3_rag.ipynb)

In this lab you build a **Retrieval-Augmented Generation (RAG)** system end to end — a question-answering tool grounded in real drug-label documents.

**How to use it**
- Run each cell in order (`Shift + Enter`).
- Read the output *and* the explanation around it.
- Cells marked **🔧 YOUR TURN** are for you to change a value and re-run.
- You will run **Parts 1–3** in the short warm-up after the Block E lecture, then **Parts 4–7** in Block F.

**What you will build**
1. **Chunk** drug-label documents into passages
2. **Embed** the chunks and build a searchable **index**
3. **Retrieve** the most relevant chunks for a question
4. **Generate** a grounded answer — *with citations*
5. Put it together into a full **RAG pipeline**
6. **Break it on purpose** — see the failure modes
7. Inspect whether the answer is **faithful** to its sources

**The data:** real drug labels from the U.S. FDA's **openFDA** service — public-domain (CC0). No patient data is involved.

## Setup

Run the cells below. The first installs two libraries; the second connects everything and loads the documents.

Your Gemini API key must be available:

- **In Colab:** stored as a Secret named `GEMINI_API_KEY` (the 🔑 icon in the left sidebar, with *Notebook access* switched on).
- **Running locally:** set as an environment variable before launching Jupyter (`export GEMINI_API_KEY=...` on macOS/Linux, `$env:GEMINI_API_KEY = "..."` on Windows PowerShell).

In [1]:
!pip install -q google-genai sentence-transformers

In [2]:
import json
import os
import re
import numpy as np
import pandas as pd
from google import genai
from sentence_transformers import SentenceTransformer

# --- Read the Gemini API key (Colab Secrets, or local .env / env var) ---
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY not found. In Colab, store it as a Secret named "
        "GEMINI_API_KEY. Locally, copy .env.example to .env and paste your "
        "key there (or set the environment variable before launching Jupyter)."
    )

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-3.1-flash-lite"

# --- The embedding model (for retrieval) — the biomedical model from Lab 1 ---
embedder = SentenceTransformer("NeuML/pubmedbert-base-embeddings")

# --- Course data lives in the GitHub repo ---
BASE_URL = "https://raw.githubusercontent.com/angeloziletti/nlp-cas-dsmh/main/data/"

print("Setup complete. Generation model:", MODEL)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Setup complete. Generation model: gemini-3.1-flash-lite


In [3]:
# Load the drug-label corpus. Each "document" is one section of one drug's label.
corpus = pd.read_json(BASE_URL + "drug_labels.json").to_dict("records")

drugs = sorted(set(d["drug"] for d in corpus))
print(f"Loaded {len(corpus)} documents covering {len(drugs)} drugs.")
print("Drugs:", ", ".join(drugs))
print("\nExample document:")
print(f"  drug: {corpus[0]['drug']}   section: {corpus[0]['section']}")
print(f"  text: {corpus[0]['text'][:300]} ...")

Loaded 115 documents covering 20 drugs.
Drugs: albuterol, amlodipine, amoxicillin, aspirin, atorvastatin, azithromycin, ciprofloxacin, furosemide, gabapentin, hydrochlorothiazide, ibuprofen, levothyroxine, lisinopril, losartan, metformin, metoprolol, omeprazole, prednisone, sertraline, warfarin

Example document:
  drug: metformin   section: Indications and Usage
  text: Metformin hydrochloride tablets are indicated as an adjunct to diet and exercise to improve glycemic control in adults and pediatric patients 10 years of age and older with type 2 diabetes mellitus. Metformin hydrochloride tablets are biguanide indicated as an adjunct to diet and exercise to improve ...


---
## Part 1 — Chunking: splitting documents into passages

A whole drug-label section is too long to be one unit of retrieval — a single vector for it would be *blurry* (Block E). So the first step is **chunking**: splitting each document into smaller passages, each of which becomes one searchable unit.

Our chunker splits text into sentences, then groups sentences into chunks of roughly a target size — with a one-sentence **overlap** so a key sentence is never stranded at a boundary.

In [4]:
def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.])\s+", text) if s.strip()]

def chunk_text(text, target_chars=500, overlap_sentences=1):
    sents = split_sentences(text)
    if not sents:
        return []
    chunks, i = [], 0
    while i < len(sents):
        current, j = [], i
        while j < len(sents) and (not current or sum(len(s) for s in current) < target_chars):
            current.append(sents[j])
            j += 1
        chunks.append(" ".join(current))
        if j >= len(sents):
            break
        i = max(j - overlap_sentences, i + 1)   # step forward, keeping an overlap
    return chunks

# See it work on one document.
doc = corpus[0]
pieces = chunk_text(doc["text"])
print(f"Document: {doc['drug']} — {doc['section']}  ({len(doc['text'])} chars)")
print(f"Split into {len(pieces)} chunks:\n")
for n, p in enumerate(pieces, 1):
    print(f"--- chunk {n} ({len(p)} chars) ---\n{p}\n")

Document: metformin — Indications and Usage  (407 chars)
Split into 1 chunks:

--- chunk 1 (407 chars) ---
Metformin hydrochloride tablets are indicated as an adjunct to diet and exercise to improve glycemic control in adults and pediatric patients 10 years of age and older with type 2 diabetes mellitus. Metformin hydrochloride tablets are biguanide indicated as an adjunct to diet and exercise to improve glycemic control in adults and pediatric patients 10 years of age and older with type 2 diabetes mellitus.



Notice each chunk is a few sentences long, and consecutive chunks **share a sentence** — that is the overlap. A question can now match the *specific passage* that answers it, not a whole label section.

In [5]:
# 🔧 YOUR TURN: change the target chunk size and watch the number of chunks change.
my_target = 300

pieces = chunk_text(corpus[0]["text"], target_chars=my_target)
print(f"target_chars = {my_target}  ->  {len(pieces)} chunks")

target_chars = 300  ->  1 chunks


Now chunk the **whole corpus**. Each chunk keeps its `drug` and `section` — that metadata is what lets the system **cite its source** later.

In [6]:
chunks = []
for doc in corpus:
    for piece in chunk_text(doc["text"]):
        chunks.append({"text": piece, "drug": doc["drug"], "section": doc["section"]})

print(f"The {len(corpus)} documents became {len(chunks)} chunks.")
print("Example chunk:", {"drug": chunks[0]["drug"], "section": chunks[0]["section"]})

The 115 documents became 286 chunks.
Example chunk: {'drug': 'metformin', 'section': 'Indications and Usage'}


### Dataset statistics — why 286 chunks from 115 documents?

Each "document" here is **one section of one drug label** (e.g. `metformin / Contraindications`), not a whole label — and FDA-label sections vary wildly in length. Short sections like *Indications* often fit in a single 500-char chunk; long sections like *Warnings and Precautions* split into several. The cell below shows the distributions so you can see exactly where the 286 comes from.

In [7]:
import statistics
from collections import Counter

# --- Document-level statistics (one "document" = one section of one drug label) ---
doc_chars = [len(d["text"]) for d in corpus]
doc_sents = [len(split_sentences(d["text"])) for d in corpus]
sent_chars = [len(s) for d in corpus for s in split_sentences(d["text"])]

print("Documents (drug-label sections):")
print(f"  count             : {len(corpus)}")
print(f"  chars per doc     : min {min(doc_chars)}   median {int(statistics.median(doc_chars))}   max {max(doc_chars)}   mean {int(statistics.mean(doc_chars))}")
print(f"  sentences per doc : min {min(doc_sents)}   median {int(statistics.median(doc_sents))}   max {max(doc_sents)}   mean {statistics.mean(doc_sents):.1f}")
print(f"  chars per sentence: min {min(sent_chars)}  median {int(statistics.median(sent_chars))}  max {max(sent_chars)}  mean {int(statistics.mean(sent_chars))}")

# --- Chunk-level statistics ---
chunk_chars = [len(c["text"]) for c in chunks]
chunks_per_doc = Counter((c["drug"], c["section"]) for c in chunks)
counts = list(chunks_per_doc.values())

print("\nChunks (target_chars=500, overlap_sentences=1):")
print(f"  count             : {len(chunks)}")
print(f"  chars per chunk   : min {min(chunk_chars)}   median {int(statistics.median(chunk_chars))}   max {max(chunk_chars)}   mean {int(statistics.mean(chunk_chars))}")
print(f"  chunks per doc    : min {min(counts)}   median {int(statistics.median(counts))}   max {max(counts)}   mean {statistics.mean(counts):.2f}")

# --- How many docs end up as 1, 2, 3, ... chunks? ---
dist = Counter(counts)
print("\n  chunks-per-doc distribution:")
for k in sorted(dist):
    print(f"    {k:2d} chunk(s):  {dist[k]:3d} docs  {'#' * dist[k]}")

Documents (drug-label sections):
  count             : 115
  chars per doc     : min 128   median 1298   max 1399   mean 1093
  sentences per doc : min 1   median 8   max 46   mean 7.6
  chars per sentence: min 1  median 118  max 1134  mean 143

Chunks (target_chars=500, overlap_sentences=1):
  count             : 286
  chars per chunk   : min 109   median 542   max 1248   mean 540
  chunks per doc    : min 1   median 3   max 4   mean 2.49

  chunks-per-doc distribution:
     1 chunk(s):   24 docs  ########################
     2 chunk(s):   18 docs  ##################
     3 chunk(s):   66 docs  ##################################################################
     4 chunk(s):    7 docs  #######


In [8]:
# --- See the overlap in action: pick the longest document and show its chunks ---
longest = max(corpus, key=lambda d: len(d["text"]))
pieces = chunk_text(longest["text"])
n_sents = len(split_sentences(longest["text"]))
print(f"Longest document: {longest['drug']} - {longest['section']}")
print(f"  {len(longest['text'])} chars, {n_sents} sentences  ->  {len(pieces)} chunks\n")

# Show the boundary between consecutive chunks: the last sentence of chunk N
# should reappear as the first sentence of chunk N+1.
for n in range(len(pieces) - 1):
    last_sent = split_sentences(pieces[n])[-1]
    first_sent_next = split_sentences(pieces[n + 1])[0]
    shared = "SHARED" if last_sent == first_sent_next else "different"
    print(f"chunk {n+1} -> chunk {n+2}  [{shared}]")
    print(f"  last sentence of chunk {n+1}: {last_sent[:140]}...")
    print(f"  first sentence of chunk {n+2}: {first_sent_next[:140]}...\n")

Longest document: metformin - Dosage and Administration
  1399 chars, 3 sentences  ->  2 chunks

chunk 1 -> chunk 2  [different]
  last sentence of chunk 1: Adult Dosage for metformin hydrochloride tablets: Starting dose: 500 mg orally twice a day or 850 mg once a day, with meals Increase the dos...
  first sentence of chunk 2: Increase the dose in increments of 500 mg weekly or 850 mg every 2 weeks on the basis of glycemic control and tolerability, up to a maximum ...



The shared sentence is the **overlap**. It is controlled by the `overlap_sentences=1` argument of `chunk_text` defined above — change it to `0` (no overlap) or `2` (more overlap) and re-run the corpus chunking cell to see the chunk count shift.

---
## Part 2 — Embedding and the vector index

Now we turn every chunk into a vector — the embeddings from Lab 1 — and collect them into one matrix. That matrix **is** our vector index: the thing we search.

The **encoder** is `NeuML/pubmedbert-base-embeddings`, a BERT model pretrained on biomedical text. It maps any string to a **768-dimensional vector** — one point in a 768-d space whose geometry roughly captures meaning. There is no *decoder*: vectors are a one-way map (text → vector, never back). To recover a chunk's text we keep the `chunks` list of dicts alongside the matrix, indexed by the same row number.

In [9]:
# Embed each chunk together with its drug and section label, so the
# vector captures WHAT the chunk is about - not just its raw words.
embed_inputs = [f"{c['drug']}, {c['section']}. {c['text']}" for c in chunks]
chunk_vectors = embedder.encode(embed_inputs, show_progress_bar=True)

print(f"Embedded {len(chunks)} chunks.")
print(f"The index is a matrix of shape {chunk_vectors.shape}  (chunks x vector-dimensions).")

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embedded 286 chunks.
The index is a matrix of shape (286, 768)  (chunks x vector-dimensions).


Notice we embedded each chunk **with its drug and section label attached** — *"metformin, Contraindications. ..."* — not just the bare text. A bare passage does not say which drug it belongs to; attaching the label so the embedding captures it is a small, standard trick that makes retrieval noticeably more accurate.

That matrix is our **vector index**. We search it by brute force — comparing a query against every chunk — exactly as in Lab 1. At this scale (a few hundred chunks) brute force is instant; a production system with millions of chunks would use a dedicated **vector database** (such as FAISS or Chroma) that does the same nearest-neighbour search efficiently.

---
## Part 3 — Retrieval: finding the right chunks

Retrieval finds the chunks most relevant to a question. We combine **two signals**:

- **Semantic similarity** — meaning-match, using the embeddings (as in Lab 1). One line of numpy — `chunk_vectors @ qv / (||rows|| · ||qv||)` — scores all chunks against the query at once: a matrix-multiply, then divide by L2 norms = cosine similarity.
- **Keyword overlap** — exact-term match, so a question naming *"metformin"* is pulled toward chunks that literally contain *metformin*.

Blending the two — a **hybrid** search — is a standard, practical way to make retrieval more reliable than meaning-match alone. We return the best `k` chunks.

In [10]:
STOPWORDS = {"what", "are", "the", "for", "and", "with", "that", "this", "can",
             "does", "have", "has", "its", "from", "any", "not", "you", "your",
             "when", "which", "how", "why", "there", "their", "they", "into",
             "about", "should", "would", "use", "used", "using", "most", "common"}

def tokenize(text):
    return {w for w in re.findall(r"[a-z]{3,}", text.lower()) if w not in STOPWORDS}

# keyword set for every chunk (drug + section + body text)
chunk_keywords = [tokenize(f"{c['drug']} {c['section']} {c['text']}") for c in chunks]

def retrieve(query, k=6):
    # signal 1 - semantic similarity (meaning)
    qv = embedder.encode(query)
    dense = chunk_vectors @ qv / (np.linalg.norm(chunk_vectors, axis=1) * np.linalg.norm(qv))
    dense = (dense - dense.min()) / (dense.max() - dense.min())   # scale to 0-1
    # signal 2 - keyword overlap (exact terms)
    q_words = tokenize(query)
    lexical = np.array([len(q_words & kw) / max(len(q_words), 1) for kw in chunk_keywords])
    # blend the two signals
    score = 0.6 * dense + 0.4 * lexical
    top = np.argsort(score)[::-1][:k]
    return [(chunks[i], float(score[i])) for i in top]

# Try it.
for chunk, score in retrieve("What are the adverse reactions to atorvastatin?"):
    print(f"({score:.3f})  {chunk['drug']} - {chunk['section']}")
    print(f"   {chunk['text'][:150]} ...\n")

(1.000)  atorvastatin - Adverse Reactions
   The following important adverse reactions are described below and elsewhere in the labeling: Myopathy and Rhabdomyolysis Immune-Mediated Necrotizing M ...

(0.959)  atorvastatin - Adverse Reactions
   To report SUSPECTED ADVERSE REACTIONS, contact Novadoz Pharmaceuticals LLC at 1-855-668-2369 or FDA at 1-800-FDA-1088 or www.fda.gov/medwatch. 6.1 Cli ...

(0.884)  atorvastatin - Adverse Reactions
   In the atorvastatin calcium placebo-controlled clinical trial database of 16,066 patients (8755 atorvastatin calcium vs. 7311 placebo; age range 10 to ...

(0.818)  metoprolol - Adverse Reactions
   The following adverse reactions are described elsewhere in labeling: Worsening angina or myocardial infarction Worsening heart failure . Worsening AV  ...

(0.815)  lisinopril - Adverse Reactions
   Common adverse reactions (events 2% greater than placebo) by use: Hypertension: headache, dizziness and cough Heart Failure: hypotension and chest pai ...


Look at what came back: the chunks most similar in meaning to the question, each tagged with the drug and section it came from. **This is the moment to inspect the index** — are these chunks actually relevant? Their scores tell you how confident the match is.

> This is the end of the Block E warm-up. In Block F, we continue from Part 4.

In [11]:
# 🔧 YOUR TURN: retrieve chunks for your own question.
my_query = "What are the side effects of statins?"

for chunk, score in retrieve(my_query):
    print(f"({score:.3f})  {chunk['drug']} — {chunk['section']}")

(0.731)  azithromycin — Adverse Reactions
(0.723)  amlodipine — Adverse Reactions
(0.722)  azithromycin — Adverse Reactions
(0.684)  ibuprofen — Adverse Reactions
(0.667)  metoprolol — Adverse Reactions
(0.621)  albuterol — Drug Interactions


---
## Part 4 — Augmented generation: the grounded answer

So far we have only *retrieved* text. Now we add the **generation** step: hand the retrieved chunks to the model and ask it to answer **using only those passages**, and to **cite** them.

This is the grounded prompt from the Block E lecture — Block C prompting plus Block D's grounding rule.

In [12]:
def build_prompt(question, retrieved):
    passages = ""
    for i, (chunk, score) in enumerate(retrieved, 1):
        passages += f"Passage {i} (from {chunk['drug']}, {chunk['section']}):\n{chunk['text']}\n\n"
    return f"""You are a clinical information assistant. Answer the question using ONLY the passages below.
If the passages do not contain the answer, reply exactly: "The provided passages do not contain this information."
Cite the passage number(s) you used, like (Passage 2).

{passages}Question: {question}

Answer:"""

# Build and inspect the prompt for one question.
question = "What are the contraindications for metformin?"
retrieved = retrieve(question)
prompt = build_prompt(question, retrieved)
print(prompt)

You are a clinical information assistant. Answer the question using ONLY the passages below.
If the passages do not contain the answer, reply exactly: "The provided passages do not contain this information."
Cite the passage number(s) you used, like (Passage 2).

Passage 1 (from metformin, Contraindications):
Metformin hydrochloride tablets are contraindicated in patients with: Severe renal impairment (eGFR below 30 mL/min/1.73 m 2 ) [ see Warnings and Precautions ]. Hypersensitivity to metformin. Acute or chronic metabolic acidosis, including diabetic ketoacidosis, with or without coma. Severe renal impairment (eGFR below 30 mL/min/1.73 m 2 ) ( 4 , 5.1 ) Hypersensitivity to metformin Acute or chronic metabolic acidosis, including diabetic ketoacidosis, with or without coma.

Passage 2 (from metformin, Drug Interactions):
Intervention: Consider more frequent monitoring of these patients. Examples: Topiramate, zonisamide, acetazolamide or dichlorphenamide. Drugs that Reduce metformin hy

In [ ]:
# Send the grounded prompt to the model.
answer = client.models.generate_content(model=MODEL, contents=prompt).text
print(answer)

Read the answer against the passages above it. The model answered **from the retrieved text**, and the citation — *(Passage 1)* — points back to a specific drug-label section a human can open and verify. That is the whole point of RAG: an answer you can **check**.

---
## Part 5 — The full RAG pipeline

Wrap retrieval and generation into one function — a complete RAG system in a few lines.

In [14]:
def rag_answer(question, k=6):
    retrieved = retrieve(question, k)
    prompt = build_prompt(question, retrieved)
    answer = client.models.generate_content(model=MODEL, contents=prompt).text
    return answer, retrieved

for q in ["Can warfarin be taken during pregnancy?",
          "What is the recommended use of amoxicillin?",
          "What should be monitored in a patient taking furosemide?"]:
    answer, retrieved = rag_answer(q)
    sources = ", ".join(sorted(set(f"{c['drug']}/{c['section']}" for c, s in retrieved)))
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"   [retrieved from: {sources}]\n")

Task was destroyed but it is pending!
task: <Task pending name='Task-155' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\angel\anaconda3\envs\nlp-cas-dsmh\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-156' coro=<Kernel.shell_main() running at C:\Users\angel\anaconda3\envs\nlp-cas-dsmh\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\angel\anaconda3\envs\nlp-cas-dsmh\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
C:\Users\angel\anaconda3\envs\nlp-cas-dsmh\Lib\site-packages\torch\nn\modules\module.py:932: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  for module in self.children():
Task was destroyed but it is pending!
task: <Task pending name='Task-156' coro=<Kernel.shell_main() running at C:\Users\angel\anaconda3\envs\nlp-cas-dsmh\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]>


Q: Can warfarin be taken during pregnancy?
A: Warfarin sodium is contraindicated in pregnant women, except in pregnant women with mechanical heart valves, who are at high risk of thromboembolism (Passage 2). While warfarin sodium may cause fetal harm, the benefits may outweigh the risks in pregnant women with mechanical heart valves (Passage 3). Exposure during pregnancy causes a recognized pattern of major congenital malformations (warfarin embryopathy and fetotoxicity), fatal fetal hemorrhage, and an increased risk of spontaneous abortion and fetal mortality (Passage 1, Passage 2).
   [retrieved from: hydrochlorothiazide/Indications and Usage, lisinopril/Warnings and Precautions, warfarin/Contraindications, warfarin/Warnings and Precautions]

Q: What is the recommended use of amoxicillin?
A: The recommended use of amoxicillin includes the following:

*   **Dosing for Adults:** 750 to 1750 mg/day in divided doses every 8 to 12 hours (Passage 6).
*   **Dosing for Pediatric Patients ove

In [15]:
# 🔧 YOUR TURN: ask the RAG system your own question about any of the 20 drugs.
my_question = "What are the warnings for ibuprofen?"

answer, retrieved = rag_answer(my_question)
print(answer)

Warnings for ibuprofen include:

*   **Cardiovascular Risks:** Clinical trials have shown an increased risk of serious cardiovascular (CV) thrombotic events, including myocardial infarction (MI) and stroke, which can be fatal. This risk has been observed most consistently at higher doses. The relative increase in these events appears to be similar in those with and without known CV disease or risk factors. Patients and physicians should remain alert for these events throughout the entire treatment course, even if no previous CV symptoms existed (Passage 5, Passage 6).
*   **Dosing/Usage:** Use the lowest effective dose for the shortest duration consistent with individual patient treatment goals (Passage 1, Passage 2, Passage 5). Do not exceed 3200 mg total daily dose (Passage 2).
*   **Hypersensitivity/Allergies:** Ibuprofen is contraindicated in patients with known hypersensitivity to it. It should not be given to patients who have experienced asthma, urticaria, or allergic-type react

---
## Part 6 — Try to Break it 

A RAG system is only as good as its retrieval and its corpus (Block E). Let's make it fail — deliberately — so the failure modes are real, not theoretical.

**Failure 1 — a question the corpus cannot answer.** Our corpus has 20 drugs. Ask about one that is *not* in it.

In [16]:
# Insulin is NOT in our 20-drug corpus.
answer, retrieved = rag_answer("What are the contraindications for insulin?")
print("ANSWER:")
print(answer)
print("\nWHAT WAS RETRIEVED:")
for chunk, score in retrieved:
    print(f"  ({score:.3f})  {chunk['drug']} — {chunk['section']}")

ANSWER:
The provided passages do not contain this information.

WHAT WAS RETRIEVED:
  (0.800)  omeprazole — Contraindications
  (0.754)  losartan — Contraindications
  (0.729)  omeprazole — Contraindications
  (0.632)  furosemide — Contraindications
  (0.630)  lisinopril — Contraindications
  (0.614)  ibuprofen — Contraindications


**Inspect what happened.** Retrieval *always* returns the closest chunks — even when nothing is truly relevant, it returns *something*. The question is what the model did with weak passages:

- A well-behaved RAG system says *"the provided passages do not contain this information"* — the grounding rule working.
- A combination of poorly grounded RAG + a bad model would answer anyway, possibly from memory.

In [20]:
# Failure 2 — an off-topic query. Retrieval still returns something.
print("Query: 'What is the capital of France?'\n")
for chunk, score in retrieve("What is the capital of France?"):
    print(f"  ({score:.3f})  {chunk['drug']} — {chunk['section']}")

Query: 'What is the capital of France?'

  (0.600)  omeprazole — Adverse Reactions
  (0.591)  metformin — Dosage and Administration
  (0.576)  amlodipine — Adverse Reactions
  (0.572)  hydrochlorothiazide — Indications and Usage
  (0.571)  ciprofloxacin — Contraindications
  (0.570)  gabapentin — Adverse Reactions


The scores are low and the chunks are irrelevant — but retrieval **still returned its top 4**. Retrieval never says "nothing found"; it always returns *the closest things it has*. A RAG system must be built to notice when the closest things are not good enough.

> **The lesson (Block E):** RAG makes answers grounded and checkable — it does not make them guaranteed correct. Retrieval can miss, and the corpus has limits.

---
## Recap — what you built

| Part | What you did |
|---|---|
| Chunking | Split documents into overlapping passages |
| Embedding & index | Turned chunks into vectors — a searchable index |
| Retrieval | Found the most relevant chunks for a question |
| Augmented generation | Generated a grounded answer with citations |
| Full pipeline | Combined it into one `rag_answer` function |
| For awereness | Saw retrieval always return *something* — and why grounding matters |